### Goal
Find the linear gain **G** of the APA_200MHz PA (OpenDPD, see `data/raw/APA_200MHz/SOURCE.md`).

G is the target of the DPD in `dpd_lstm.ipynb`: `PA(DPD(x)) ≈ G·x`. It is saved in `results/APA_200MHz/linear_gain.json`.

Documentation: `docs/codigo/apa_linear_gain.md`

#### 1. Setup

In [ ]:
# Importing packages
import json
import numpy as np
import matplotlib.pyplot as plt

from pa_model import Dataset, DATA_RAW, RESULTS

DATASET = "APA_200MHz"

#### 2. Functions

In [ ]:
EPS = 1e-15

def to_decibel(v):
    """complexo -> magnitude em dB"""
    return 20 * np.log10(np.abs(v) + EPS)

In [ ]:
# Ploting the AM/AM signal (optional: line y_hat = slope*x and the bounds of the fit range)
def plot_AM_AM(x, y, db=True, title="AM/AM Graph", title_flag="", x_label="|x|", y_label="|y|",
               slope=None, fit_range=None):
    fig, ax = plt.subplots(figsize=(10, 5))
    if db:
        ax.plot(to_decibel(x), to_decibel(y), '.', ms=3, alpha=.3)
        ax.set_xlabel(f"{x_label} (dB)"); ax.set_ylabel(f"{y_label} (dB)")
    else:
        ax.plot(np.abs(x), np.abs(y), '.', ms=3, alpha=.3)
        ax.set_xlabel(x_label); ax.set_ylabel(y_label)

    if slope is not None:
        x_line = np.linspace(0, np.max(np.abs(x)), 200)
        if db:
            # In dB the line y = m*x becomes y_dB = x_dB + 20*log10(m)
            ax.plot(to_decibel(x_line[1:]), to_decibel(slope * x_line[1:]), 'r--', lw=1.5, label=f"ŷ = {slope:.4f}·x")
        else:
            ax.plot(x_line, slope * x_line, 'r--', lw=1.5, label=f"ŷ = {slope:.4f}·x")
        ax.legend(frameon=False, loc="upper left")

    if fit_range is not None:
        for bound in fit_range:
            if db and bound <= 0:
                continue   # |x| = 0 has no dB value (it would become -300 dB because of EPS)
            ax.axvline(to_decibel(bound) if db else bound, color="k", ls=":", lw=1)

    ax.set_title(f"{title}{title_flag}"); ax.grid(True, alpha=.3)
    plt.show()

In [ ]:
# Ploting the AM/AM normalized by the peak (|x|/max|x| and |y|/max|y|) in a square figure with
# the same scale on both axes, so a linear relation between x and y shows up as a 45° line
def plot_normalized_AM_AM(x, y, db=True, title="AM/AM Graph (Normalized)"):
    x_norm = x / np.max(np.abs(x))
    y_norm = y / np.max(np.abs(y))

    if db:
        x_plot, y_plot, unit = to_decibel(x_norm), to_decibel(y_norm), " (dB)"
    else:
        x_plot, y_plot, unit = np.abs(x_norm), np.abs(y_norm), ""

    # Same limits on both axes, covering the data of both
    lo = min(x_plot.min(), y_plot.min())
    hi = max(x_plot.max(), y_plot.max())
    margin = .02 * (hi - lo)

    fig, ax = plt.subplots(figsize=(7, 7))
    ax.plot(x_plot, y_plot, '.', ms=3, alpha=.3)
    ax.set_xlim(lo - margin, hi + margin); ax.set_ylim(lo - margin, hi + margin)
    ax.set_aspect("equal")
    ax.set_xlabel(f"|x|/max|x|{unit}"); ax.set_ylabel(f"|y|/max|y|{unit}")
    ax.set_title(title); ax.grid(True, alpha=.3)
    plt.show()

In [ ]:
# Gain compression point technique
# Step 1: split the samples into bands of input amplitude |x| (dB)
def getting_samples(x, y, bw=0.5):
    x_db = to_decibel(x)

    # Grid aligned to multiples of bw; the +1 puts the last edge strictly above the max
    start = np.floor(x_db.min() / bw) * bw
    n_bands = int(np.floor(x_db.max() / bw) - np.floor(x_db.min() / bw)) + 1
    edges = start + bw * np.arange(n_bands + 1)

    # Band index of each sample: k such that edges[k] <= x_db < edges[k+1]
    idx = np.digitize(x_db, edges) - 1

    bands = [(x[idx == k], y[idx == k]) for k in range(n_bands)]
    return edges, bands

In [ ]:
# Step 2: complex gain of each band (least squares), with its center and number of samples
def band_gain(edges, bands):
    centers = (edges[:-1] + edges[1:]) / 2
    counts = np.array([len(x_k) for x_k, _ in bands])

    # Empty bands keep NaN, so position k is always band k
    G = np.full(len(bands), np.nan, dtype=complex)
    for k, (x_k, y_k) in enumerate(bands):
        if counts[k] > 0:
            G[k] = np.sum(y_k * np.conj(x_k)) / np.sum(np.abs(x_k)**2)

    gain_db = 20 * np.log10(np.abs(G))
    phase_deg = np.angle(G, deg=True)

    return centers, counts, gain_db, phase_deg

In [ ]:
# Step 3: Ploting the gain curve: |G_k|, angle G_k and number of samples per band
def plot_band_gain(centers, counts, gain_db, phase_deg, min_count=50, title="Gain per Band"):
    # Lower limit: bands with fewer than min_count samples are dominated by noise and are not drawn
    valid = counts >= min_count

    fig, (ax_g, ax_p, ax_n) = plt.subplots(3, 1, figsize=(10, 9), sharex=True, gridspec_kw={"height_ratios": [2, 2, 1]}, constrained_layout=True)

    ax_g.plot(centers, np.where(valid, gain_db, np.nan), 'o-', ms=3, lw=1)
    ax_g.set_ylabel("|G_k| (dB)")

    ax_p.plot(centers, np.where(valid, phase_deg, np.nan), 'o-', ms=3, lw=1)
    ax_p.set_ylabel("∠G_k (°)")

    # Log scale: counts go from 1 to thousands; empty bands are not drawn; discarded bands in gray
    width = centers[1] - centers[0]
    n = np.where(counts > 0, counts, np.nan)
    ax_n.bar(centers[valid], n[valid], width=width * .9, label=f"used (≥ {min_count})")
    ax_n.bar(centers[~valid], n[~valid], width=width * .9, color="tab:gray", alpha=.5, label="discarded")
    ax_n.axhline(min_count, color="k", ls="--", lw=.8)
    ax_n.set_yscale("log")
    ax_n.set_ylabel("samples"); ax_n.set_xlabel("|x| (dB)")
    ax_n.legend(frameon=False, loc="upper left")

    for ax in (ax_g, ax_p, ax_n):
        ax.grid(True, alpha=.3); ax.set_axisbelow(True)

    fig.suptitle(title)
    plt.show()

In [ ]:
# Least squares line through the origin, y_hat = m*x, on the magnitudes: m = sum(|x||y|) / sum(|x|^2)
# Only the samples with x_min < |x| < x_max are used in the fit
def fit_slope(x, y, x_min=0, x_max=0.7):
    ax_, ay_ = np.abs(x), np.abs(y)
    mask = (ax_ > x_min) & (ax_ < x_max)
    m = np.sum(ax_[mask] * ay_[mask]) / np.sum(ax_[mask]**2)
    return m, mask

In [ ]:
# Calculate the Linear Gain
def get_linear_gain(x,y):
    xi, xq = x.real, x.imag
    yi, yq = y.real, y.imag

    den = np.sum(xi**2 + xq**2)
    gi = np.sum(yi*xi + yq*xq)/den
    gq = np.sum(yq*xi - yi*xq)/den

    return gi, gq

In [ ]:
# Calculate the linear output
def get_linear_output(x, gi, gq):
    xi, xq = x.real, x.imag

    yi_lin = gi*xi - gq*xq
    yq_lin = gi*xq + gq*xi

    return yi_lin, yq_lin

In [ ]:
# Comparing the real output with the expected linear output
def plot_real_vs_linear(y, y_lini, y_linq, title="Real vs Linear Output"):
    fig, axes = plt.subplots(1, 2, figsize=(10, 4.8), constrained_layout=True)

    # I and Q against the ideal linear response
    for ax, real, lin, name in ((axes[0], y.real, y_lini, "I"),
                                (axes[1], y.imag, y_linq, "Q")):
        lo, hi = min(lin.min(), real.min()), max(lin.max(), real.max())
        ax.plot(lin, real, '.', ms=2, alpha=.2, color="#2a78d6", rasterized=True)
        ax.plot([lo, hi], [lo, hi], '-', color="#898781", lw=1.5, label="ideal linear")
        ax.set_xlabel(f"$Y_{{lin,{name}}}$"); ax.set_ylabel(f"$Y_{{real,{name}}}$")
        ax.set_title(f"{name} component"); ax.set_aspect('equal', 'box')
        ax.grid(True, alpha=.3); ax.set_axisbelow(True)
        ax.legend(frameon=False, loc="upper left")

    fig.suptitle(title)
    plt.show()

In [ ]:
# Gain deviation: how far the real gain sits from the fitted linear gain, per input level
def plot_gain_deviation(x, y, y_lini, y_linq, title="Gain Deviation"):
    x_db = to_decibel(x)
    dev = to_decibel(y) - to_decibel(y_lini + 1j*y_linq)

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.axhline(0, color="#898781", lw=1.5, label="ideal linear")
    ax.plot(x_db, dev, '.', ms=2, alpha=.2, color="#2a78d6", rasterized=True)
    ax.set_xlabel("|x| (dB)"); ax.set_ylabel(r"$|Y_{real}| - |Y_{lin}|$ (dB)")
    ax.set_xlim(np.percentile(x_db, .1), x_db.max() + 1)
    ax.set_title(title); ax.grid(True, alpha=.3); ax.set_axisbelow(True)
    ax.legend(frameon=False, loc="upper left")
    plt.show()

#### 3. Loading Data

In [ ]:
# Loading the dataset: the gain uses all the samples, so train, val and test files are joined back together
parts = Dataset.from_split_csv(DATA_RAW / DATASET)
ds = Dataset(np.concatenate([p.x for p in parts]), np.concatenate([p.y for p in parts]))

print(f"{len(ds)} samples, max|x| = {np.abs(ds.x).max():.3f}, max|y| = {np.abs(ds.y).max():.3f}")

#### 4. AM/AM

In [ ]:
# Ploting AM/AM (dB and linear)
plot_AM_AM(ds.x, ds.y, title_flag=f" [{DATASET}]")
plot_AM_AM(ds.x, ds.y, db=False, title_flag=f" [{DATASET}]")

In [ ]:
# Ploting AM/AM normalized by the peak (square, same scale on both axes)
plot_normalized_AM_AM(ds.x, ds.y, title=f"AM/AM Graph (Normalized) [{DATASET}]")
plot_normalized_AM_AM(ds.x, ds.y, db=False, title=f"AM/AM Graph (Normalized) [{DATASET}]")

#### 5. Gain per Band (gain compression point technique)
Bands of input amplitude |x| (dB), complex gain of each band by least squares, and the gain curve. Bands with fewer than `MIN_COUNT` samples are dominated by noise and are discarded.

In [ ]:
# Testing getting_samples function
edges, bands = getting_samples(ds.x, ds.y, bw=0.5)
counts = np.array([len(xk) for xk, _ in bands])

print(f"|x| range: {to_decibel(ds.x).min():.2f} to {to_decibel(ds.x).max():.2f} dB")
print(f"edges:     {edges[0]:.2f} to {edges[-1]:.2f} dB ({len(bands)} bands of {edges[1] - edges[0]} dB)")
print(f"samples:   {counts.sum()} in bands / {len(ds.x)} total")
print(f"empty bands: {(counts == 0).sum()}")

In [ ]:
# Gain curve of the dataset (bands with fewer than MIN_COUNT samples are discarded)
MIN_COUNT = 50

centers, counts, gain_db, phase_deg = band_gain(edges, bands)
plot_band_gain(centers, counts, gain_db, phase_deg, min_count=MIN_COUNT, title=f"Gain per Band [{DATASET}]")

#### 6. Linear Gain: least squares line through the origin
`ŷ = m·x` fitted on the magnitudes, only with the samples in `X_MIN < |x| < X_MAX` (above `X_MAX` the PA compresses).

In [ ]:
# Least squares line y_hat = m*x fitted on 0 < |x| < 0.7
X_MIN, X_MAX = 0, 0.7

m, fit_mask = fit_slope(ds.x, ds.y, x_min=X_MIN, x_max=X_MAX)
print(f"m = {m:.4f} ({to_decibel(m):.3f} dB), using {fit_mask.sum()} of {len(ds.x)} samples")

plot_AM_AM(ds.x, ds.y, db=False, slope=m, fit_range=(X_MIN, X_MAX), title_flag=f" [{DATASET}, fit {X_MIN} < |x| < {X_MAX}]")

#### 7. Complex Gain G
Same samples as the line, but with complex numbers (`get_linear_gain`): G has a magnitude **and** a phase. This is the value used by the DPD.

In [ ]:
# Getting the complex linear gain on the fit range and the expected linear output
X, Y = ds.x, ds.y
gi, gq = get_linear_gain(X[fit_mask], Y[fit_mask])
G = gi + 1j*gq
Y_lini, Y_linq = get_linear_output(X, gi, gq)

print(f"G = {gi:.4f} + j{gq:.4f}  ->  |G| = {np.abs(G):.4f} ({to_decibel(G):.3f} dB), angle = {np.angle(G, deg=True):.2f} deg")
print(f"|G| x max|x| = {np.abs(G) * np.abs(X).max():.3f}  (max|y| measured = {np.abs(Y).max():.3f})")

In [ ]:
# Comparing the real output with the expected linear output
plot_real_vs_linear(Y, Y_lini, Y_linq, title=f"Real vs Linear Output [{DATASET}]")
plot_gain_deviation(X, Y, Y_lini, Y_linq, title=f"Gain Deviation [{DATASET}]")

#### 8. Saving G (read by `dpd_lstm.ipynb`)

In [ ]:
# Saving the gain next to the PA LSTM results of the same dataset
out_dir = RESULTS / DATASET
out_dir.mkdir(parents=True, exist_ok=True)

linear_gain = {
    "dataset": DATASET,
    "gi": float(gi), "gq": float(gq),
    "abs": float(np.abs(G)), "abs_db": float(to_decibel(G)), "angle_deg": float(np.angle(G, deg=True)),
    "fit_range": [X_MIN, X_MAX], "n_samples": int(fit_mask.sum()),
}
with open(out_dir / "linear_gain.json", "w") as f:
    json.dump(linear_gain, f, indent=4)

print(json.dumps(linear_gain, indent=4))